# layout
> Map dates onto a Sunday-first weekday × week grid.

In [ ]:
#| default_exp layout

In [ ]:
#| export
import calendar
from datetime import date, timedelta

In [ ]:
#| export
def weekday_row(d):
    "Row index with Sunday=0 .. Saturday=6."
    return (d.weekday() + 1) % 7

def sunday_on_or_before(d):
    "The Sunday on or before `d`."
    return d - timedelta(days=weekday_row(d))

def saturday_on_or_after(d):
    "The Saturday on or after `d`."
    return d + timedelta(days=(5 - d.weekday()) % 7)

def week_col(d, grid_start):
    "Column index of `d` measured in weeks from `grid_start` (a Sunday)."
    return (d - grid_start).days // 7

def grid_bounds(block_start, block_end):
    "Return (grid_start, n_cols) of full Sunday-first weeks covering [block_start, block_end]."
    gs = sunday_on_or_before(block_start)
    ge = saturday_on_or_after(block_end)
    return gs, (ge - gs).days // 7 + 1

In [ ]:
from fastcore.test import test_eq

# 2024-01-07 is a Sunday, 2024-01-06 a Saturday, 2023-12-31 a Sunday
test_eq(weekday_row(date(2024, 1, 7)), 0)
test_eq(weekday_row(date(2024, 1, 6)), 6)
test_eq(sunday_on_or_before(date(2024, 1, 1)), date(2023, 12, 31))
test_eq(sunday_on_or_before(date(2024, 1, 7)), date(2024, 1, 7))
test_eq(saturday_on_or_after(date(2024, 1, 1)), date(2024, 1, 6))
test_eq(saturday_on_or_after(date(2024, 1, 6)), date(2024, 1, 6))
test_eq(week_col(date(2024, 1, 7), date(2023, 12, 31)), 1)
test_eq(grid_bounds(date(2024, 1, 1), date(2024, 1, 13)), (date(2023, 12, 31), 2))

In [ ]:
#| export
def year_blocks(start, end):
    "Split [start, end] into (year, block_start, block_end); one block per calendar year."
    if start > end: raise ValueError("start must be <= end")
    if start.year == end.year: return [(start.year, start, end)]
    out = []
    for y in range(start.year, end.year + 1):
        out.append((y, max(start, date(y, 1, 1)), min(end, date(y, 12, 31))))
    return out

def month_label_cols(block_start, block_end, grid_start):
    "List of (week_col, 'Mon') for the first day of each month in [block_start, block_end]."
    out, seen, d = [], set(), block_start
    while d <= block_end:
        if (d.year, d.month) not in seen:
            seen.add((d.year, d.month))
            out.append((week_col(d, grid_start), calendar.month_abbr[d.month]))
        d += timedelta(days=1)
    return out

In [ ]:
from fastcore.test import test_eq, test_fail

# single calendar year -> one block
test_eq(year_blocks(date(2026, 1, 1), date(2026, 6, 23)),
        [(2026, date(2026, 1, 1), date(2026, 6, 23))])

# multi-year -> one block per year, clipped
b = year_blocks(date(2024, 11, 1), date(2026, 2, 1))
test_eq([y for y, _, _ in b], [2024, 2025, 2026])
test_eq(b[0], (2024, date(2024, 11, 1), date(2024, 12, 31)))
test_eq(b[1], (2025, date(2025, 1, 1), date(2025, 12, 31)))
test_eq(b[2], (2026, date(2026, 1, 1), date(2026, 2, 1)))

test_fail(lambda: year_blocks(date(2026, 2, 1), date(2026, 1, 1)), contains="start")

# month labels in order
gs = sunday_on_or_before(date(2024, 1, 1))
test_eq([l for _, l in month_label_cols(date(2024, 1, 1), date(2024, 3, 31), gs)],
        ["Jan", "Feb", "Mar"])

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()